Formative 3, Part 2: Bayesian Probability on IMDb Reviews Team 19

  |Teammates: MUTONI KEIRA  Habibllah Ayoola Ayodele Merveille Shekina Ineza
  
We apply Bayes' theorem to estimate how likely a review is positive when it contains a given keyword, using the IMDb 50K movie reviews dataset and basic Python only.

In [27]:
import csv

def load_reviews(path):
    """Read the CSV and return a list of (review_text, sentiment) pairs."""
    reviews = []
    with open(path, encoding="utf-8") as f:
        for row in csv.DictReader(f):
            reviews.append((row["review"], row["sentiment"]))
    return reviews

reviews = load_reviews("/IMDB Dataset.csv")
print("Total reviews:", len(reviews))

Total reviews: 50000


In [28]:
def tokenize(text):
    """Lowercase, remove HTML breaks and punctuation, return the set of words."""
    text = text.lower().replace("<br />", " ")
    for ch in '.,!?;:"()[]-/':
        text = text.replace(ch, " ")
    return set(text.split())

def prepare(reviews):
    """Return a list of (word_set, is_positive) pairs."""
    return [(tokenize(text), label == "positive") for text, label in reviews]

data = prepare(reviews)
print("Prepared:", len(data))
print("Is 'excellent' in the first review?", "excellent" in data[0][0])

Prepared: 50000
Is 'excellent' in the first review? False


In [29]:
def bayes_for_keyword(data, keyword):
    """Compute prior, likelihood, marginal and posterior P(Positive | keyword)."""
    total = len(data)
    n_pos = sum(1 for words, is_pos in data if is_pos)
    n_keyword = sum(1 for words, is_pos in data if keyword in words)
    n_keyword_and_pos = sum(1 for words, is_pos in data if is_pos and keyword in words)

    if n_keyword == 0:
        return None  # keyword never appears

    prior = n_pos / total                      # P(Positive)
    likelihood = n_keyword_and_pos / n_pos     # P(keyword | Positive)
    marginal = n_keyword / total               # P(keyword)
    posterior = likelihood * prior / marginal  # P(Positive | keyword)

    return {"keyword": keyword, "prior": prior, "likelihood": likelihood,
            "marginal": marginal, "posterior": posterior,
            "check": n_keyword_and_pos / n_keyword}

In [30]:
positive_keywords = ["excellent", "wonderful", "amazing"]
negative_keywords = ["terrible", "waste", "boring"]

print(f"{'keyword':<12}{'prior':>8}{'likelihood':>12}{'marginal':>10}{'posterior':>11}{'check':>8}")
for kw in positive_keywords + negative_keywords:
    r = bayes_for_keyword(data, kw)
    if r:
        print(f"{kw:<12}{r['prior']:>8.4f}{r['likelihood']:>12.4f}{r['marginal']:>10.4f}{r['posterior']:>11.4f}{r['check']:>8.4f}")

keyword        prior  likelihood  marginal  posterior   check
excellent     0.5000      0.1145    0.0709     0.8074  0.8074
wonderful     0.5000      0.0902    0.0555     0.8120  0.8120
amazing       0.5000      0.0671    0.0431     0.7783  0.7783
terrible      0.5000      0.0152    0.0539     0.1414  0.1414
waste         0.5000      0.0070    0.0506     0.0687  0.0687
boring        0.5000      0.0236    0.0610     0.1936  0.1936


In [31]:
def show_keyword_block(result):
    """Print one small table for a single keyword."""
    print(f"Keyword: {result['keyword']}")
    print(f"  Prior      P(Positive)           = {result['prior']:.4f}")
    print(f"  Likelihood P(keyword | Positive) = {result['likelihood']:.4f}")
    print(f"  Marginal   P(keyword)            = {result['marginal']:.4f}")
    print(f"  Posterior  P(Positive | keyword) = {result['posterior']:.4f}")
    print()

for kw in positive_keywords + negative_keywords:
    show_keyword_block(bayes_for_keyword(data, kw))

Keyword: excellent
  Prior      P(Positive)           = 0.5000
  Likelihood P(keyword | Positive) = 0.1145
  Marginal   P(keyword)            = 0.0709
  Posterior  P(Positive | keyword) = 0.8074

Keyword: wonderful
  Prior      P(Positive)           = 0.5000
  Likelihood P(keyword | Positive) = 0.0902
  Marginal   P(keyword)            = 0.0555
  Posterior  P(Positive | keyword) = 0.8120

Keyword: amazing
  Prior      P(Positive)           = 0.5000
  Likelihood P(keyword | Positive) = 0.0671
  Marginal   P(keyword)            = 0.0431
  Posterior  P(Positive | keyword) = 0.7783

Keyword: terrible
  Prior      P(Positive)           = 0.5000
  Likelihood P(keyword | Positive) = 0.0152
  Marginal   P(keyword)            = 0.0539
  Posterior  P(Positive | keyword) = 0.1414

Keyword: waste
  Prior      P(Positive)           = 0.5000
  Likelihood P(keyword | Positive) = 0.0070
  Marginal   P(keyword)            = 0.0506
  Posterior  P(Positive | keyword) = 0.0687

Keyword: boring
  Prior    